# 2b. Lazy Loading (deep dive)

Trajectories from long simulations can be far larger than RAM. genepie's **lazy
loading** streams a DCD file one frame at a time, so peak memory stays flat no
matter how long the trajectory is — while producing results that are
bit-for-bit identical to the in-memory path.

This chapter explains *why* lazy loading helps, *how* it works, and shows the two
ways to use it, with a live memory measurement on the bundled BPTI system.


## The problem it solves

Eager loading (`crd_convert()` without `lazy=True`) reads the **entire**
trajectory into a NumPy array of shape `(nframe, natom, 3)`:

$$\text{memory} = n_{\text{frame}} \times n_{\text{atom}} \times 3 \times 8\ \text{bytes}.$$

Lazy loading reads one frame at a time, so peak coordinate memory is
`~1 frame` regardless of trajectory length:

| Trajectory | frames | atoms | Eager (all frames) | Lazy (one frame) |
|------------|-------:|------:|-------------------:|-----------------:|
| Bundled BPTI | 10 | 36,931 | ~8.9 MB | ~0.89 MB |
| Medium run | 100,000 | 50,000 | ~112 GB | ~1.1 MB |
| Long run | 1,000,000 | 50,000 | ~1.1 TB | ~1.1 MB |

For anything past a few GB, eager loading is simply not an option; lazy loading
makes the analysis possible on an ordinary machine.


## How it works: O(1) random access

DCD files store every frame with the **same byte size**, so frame *N* can be
reached directly with a computed offset — no scanning required:

$$\text{byte\_offset} = \text{header\_size} + (N-1)\times \text{frame\_size}.$$

GENESIS seeks to that offset with Fortran stream I/O, reads a single frame, and
feeds it into the **same** unified analysis core the CLI and the in-memory Python
path use. That shared core is why lazy and eager results match exactly.

```{mermaid}
flowchart LR
  DCD[("large.dcd on disk")] -->|"seek to byte_offset(N)"| SRC["TRJ_SOURCE_LAZY_DCD"]
  SRC -->|"one frame at a time"| CORE["analyze_*_unified() core"]
  CORE --> SINK["result array (NumPy, zerocopy)"]
```


In [ ]:
import numpy as np
from genepie import genesis_exe, SMolecule
from genepie.tests.conftest import BPTI_PDB, BPTI_PSF, BPTI_DCD

mol = SMolecule.from_file(pdb=BPTI_PDB, psf=BPTI_PSF, ref=BPTI_PDB)

## Way 1 — `crd_convert(lazy=True)` + the usual `rmsd_analysis`

Set `lazy=True` and `crd_convert` returns a *lazy* `STrajectories` that holds no
coordinates. Passing it to `rmsd_analysis` transparently streams frames from disk
— the analysis call is identical to the eager case.


In [ ]:
lazy_trajs, lazy_mol = genesis_exe.crd_convert(
    mol,
    trj_files=[str(BPTI_DCD)],
    trj_format="DCD",
    trj_type="COOR+BOX",
    selection="all",
    lazy=True,
)
lt = lazy_trajs[0]
print("is_lazy       :", lt.is_lazy)
print("coords loaded :", lt.coords)         # None -- nothing in memory
print("nframe/natom  :", lt.nframe, lt.natom)

rmsd_lazy = genesis_exe.rmsd_analysis(
    lazy_mol, lt,
    analysis_selection="an:CA",
    fitting_selection="an:CA",
    fitting_method="TR+ROT",
)
print("RMSD (way 1):", np.round(rmsd_lazy.rmsd, 3))

## Way 2 — the one-shot `rmsd_analysis_lazy`

`rmsd_analysis_lazy` wraps the whole thing in a single call that takes the DCD
path directly and also reports the DCD header info. Use `has_box=True` for
DCDs written with box information, and `max_frames` to size the output buffer.


In [ ]:
res = genesis_exe.rmsd_analysis_lazy(
    mol, str(BPTI_DCD),
    analysis_selection="an:CA",
    fitting_selection="an:CA",
    fitting_method="TR+ROT",
    has_box=True,
)
print(f"DCD header: {res.dcd_nframe} frames, {res.dcd_natom} atoms/frame")
print("RMSD (way 2):", np.round(res.rmsd, 3))

## Proof: lazy equals eager

Run the in-memory path and compare.

In [ ]:
eager_trajs, eager_mol = genesis_exe.crd_convert(
    mol, trj_files=[str(BPTI_DCD)], trj_format="DCD",
    trj_type="COOR+BOX", selection="all", lazy=False,
)
rmsd_eager = genesis_exe.rmsd_analysis(
    eager_mol, eager_trajs[0],
    analysis_selection="an:CA",
    fitting_selection="an:CA",
    fitting_method="TR+ROT",
)
np.testing.assert_allclose(rmsd_eager.rmsd, rmsd_lazy.rmsd, rtol=1e-4, atol=1e-6)
print("lazy and eager RMSD match to within 1e-4")

## See the memory saving

`tracemalloc` tracks Python (NumPy) allocations. The eager load materializes the
full `(nframe, natom, 3)` array; the lazy load allocates almost nothing.


In [ ]:
import tracemalloc

def peak_mb(fn):
    tracemalloc.start()
    fn()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1e6

eager_peak = peak_mb(lambda: genesis_exe.crd_convert(
    mol, trj_files=[str(BPTI_DCD)], trj_format="DCD",
    trj_type="COOR+BOX", selection="all", lazy=False))
lazy_peak = peak_mb(lambda: genesis_exe.crd_convert(
    mol, trj_files=[str(BPTI_DCD)], trj_format="DCD",
    trj_type="COOR+BOX", selection="all", lazy=True))

print(f"eager peak Python allocation: {eager_peak:6.2f} MB")
print(f"lazy  peak Python allocation: {lazy_peak:6.2f} MB")
print(f"reduction: {eager_peak / max(lazy_peak, 1e-6):.1f}x  (and it does not grow with trajectory length)")

## When to use it, and the restrictions

**Use lazy loading when** a trajectory is too large for RAM, or when you scan many
files and only need a scalar per frame.

**Restrictions of `crd_convert(lazy=True)`** (enforced with a clear
`GenesisValidationError`):

- a single DCD file only,
- no fitting, centering, or PBC correction *at load time* — those need all
  coordinates in memory. Do the fitting inside the analysis instead
  (`rmsd_analysis(..., fitting_selection=...)` handles it per frame).

**Knobs for `rmsd_analysis_lazy`:** `has_box` (whether the DCD carries a box) and
`max_frames` (output buffer size).

**Coverage today:** RMSD is fully lazy end-to-end. `rg_analysis` and
`drms_analysis` also accept a lazy trajectory from `crd_convert(lazy=True)`;
`trj_analysis` is a natural next candidate.
